# Hidden Markov Model for Named Entity Recognition

In [26]:
import pandas as pd
import collections

In [22]:
splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}

df_train = pd.read_parquet("hf://datasets/lhoestq/conll2003/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/lhoestq/conll2003/" + splits["test"])

In [23]:
df_train.head()

,id,tokens,pos_tags,chunk_tags,ner_tags
0,0,"[EU, rejects, German, call, to, boycott, Briti...","[22, 42, 16, 21, 35, 37, 16, 21, 7]","[11, 21, 11, 12, 21, 22, 11, 12, 0]","[3, 0, 7, 0, 0, 0, 7, 0, 0]"
1,1,"[Peter, Blackburn]","[22, 22]","[11, 12]","[1, 2]"
2,2,"[BRUSSELS, 1996-08-22]","[22, 11]","[11, 12]","[5, 0]"
3,3,"[The, European, Commission, said, on, Thursday...","[12, 22, 22, 38, 15, 22, 28, 38, 15, 16, 21, 3...","[11, 12, 12, 21, 13, 11, 11, 21, 13, 11, 12, 1...","[0, 3, 4, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, ..."
4,4,"[Germany, 's, representative, to, the, Europea...","[22, 27, 21, 35, 12, 22, 22, 27, 16, 21, 22, 2...","[11, 11, 12, 13, 11, 12, 12, 11, 12, 12, 12, 1...","[5, 0, 0, 0, 0, 3, 4, 0, 0, 0, 1, 2, 0, 0, 0, ..."


Since we only need NER tages, we can ignore POS and chunk tags.

In [24]:
df_train.drop(columns=["chunk_tags", "pos_tags"], inplace=True)
df_test.drop(columns=["chunk_tags", "pos_tags"], inplace=True)

df_train.head()

,id,tokens,ner_tags
0,0,"[EU, rejects, German, call, to, boycott, Briti...","[3, 0, 7, 0, 0, 0, 7, 0, 0]"
1,1,"[Peter, Blackburn]","[1, 2]"
2,2,"[BRUSSELS, 1996-08-22]","[5, 0]"
3,3,"[The, European, Commission, said, on, Thursday...","[0, 3, 4, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, ..."
4,4,"[Germany, 's, representative, to, the, Europea...","[5, 0, 0, 0, 0, 3, 4, 0, 0, 0, 1, 2, 0, 0, 0, ..."


#### NER Tags

0: "O" (Outside of a named entity)

1: "B-PER" (Beginning of a person's name)

2: "I-PER" (Inside of a person's name)

3: "B-ORG" (Beginning of an organization)

4: "I-ORG" (Inside of an organization)

5: "B-LOC" (Beginning of a location)

6: "I-LOC" (Inside of a location)

7: "B-MISC" (Beginning of a miscellaneous entity)

8: "I-MISC" (Inside of a miscellaneous entity)

## Model Training

### Counting Tags and Transitions

In [28]:
tag_counts = collections.defaultdict(int)
transition_counts = collections.defaultdict(int)
emission_counts = collections.defaultdict(int)

# each row represents a sentence, which contain tonkens and NER tags
for _, row in df_train.iterrows():
    tokens = row['tokens']
    tags = row['ner_tags']
    
    # starting tag
    prev_tag = "<START>"
    tag_counts[prev_tag] += 1
    
    # iterate through the words and tags at the same time
    for word, tag in zip(tokens, tags):
        # increment counts
        tag_counts[tag] += 1
        transition_counts[(prev_tag, tag)] += 1
        emission_counts[(tag, word)] += 1
        
        # use current tag for next iteration
        prev_tag = tag

### Calculating Probabilities Using Counts

In [29]:
transition_probs = {}
for (prev_tag, tag), count in transition_counts.items():
    transition_probs[(prev_tag, tag)] = count / tag_counts[prev_tag]

emission_probs = {}
for (tag, word), count in emission_counts.items():
    emission_probs[(tag, word)] = count / tag_counts[tag]

In [ ]:
for transition in list(transition_probs.items()):
    print(transition)

In [ ]:
print("\nEmissions:")
for emission in list(emission_probs.items()):
    print(emission)